# MB12 — Retrospective benchmark diagnostics and provenance audit

Uses the new wd0.1 model. C1–C4 and M1–M4 are recomputed on validation and the previously used test split with repaired paired scrubbing and distinct component alternatives. This is retrospective evaluation; no fresh held-out confirmation is claimed. Historical wd0.01 test results are archived, not reused as outputs. No S1–S3 lock is invented: corrected steering has exploratory validation in MB11c.

In [1]:
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = os.environ.get("MI_CPU_THREADS", "4")
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: True | RUN_MODE: full
ROOT: /content/soccer_mi
Track A (read only): /content/soccer_mi/model_training_evaluation | exists: True
Track B: /content/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.conflict as conflict
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.circuits as circuits
for module in [data, metrics, training, paths, fingerprint, conflict, mi_setup, hooks, interventions, mi_metrics, circuits]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Device: cuda | outputs: /content/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())

circuit = mi_setup.load_json(OUT / "config" / "circuit.json")
CIRCUIT = circuit["components"]
ALL = circuits.all_components(N_LAYERS, N_HEADS)
partners = pd.read_parquet(POP / "swap_partners.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
donors = pd.read_parquet(POP / "context_donors.parquet")
reference = torch.load(POP / "reference_means.pt", map_location="cpu", weights_only=False)
MEANS = {site: value.to(DEVICE) for site, value in reference["means"].items()}
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}
dedicated_players = np.sort(partners["player"].to_numpy())
idplayer_state = torch.load(OUT / "artifacts" / "conflict" / "references" / "id_player_seed0.pt", map_location="cpu", weights_only=False)
idplayer = conflict.IdPlayer(idplayer_state["vocabulary_sizes"]); idplayer.load_state_dict(idplayer_state["state"])
with torch.no_grad():
    all_players = torch.arange(idplayer_state["vocabulary_sizes"]["player"])
    habit_log_probs = torch.log_softmax(idplayer(all_players.unsqueeze(1), torch.zeros(len(all_players), 0))
                                        / idplayer_state["temperature"], dim=-1).numpy()
RESULTS_PATH = OUT / "results" / "MB12_test_results.json"
print(f"circuit: {CIRCUIT} ({circuit['source']}) | test results already saved: {RESULTS_PATH.exists()}")

hypothesis_path = OUT / "config" / "mechanism_hypothesis.json"
hypothesis = mi_setup.load_json(hypothesis_path) if hypothesis_path.exists() else {"nodes": {}}
CLASS_KEYS = {component: hypothesis["nodes"][component]["class_keys"]
              for component in CIRCUIT if component in hypothesis.get("nodes", {})}
SCRUB_AT_TARGET = hypothesis.get("scrub_scope", "target") == "target"
SCRUB_DRAWS = SETTINGS["scrub_draws"]
steering_path = OUT / "config" / "steering_prediction.json"
STEERING = mi_setup.load_json(steering_path) if steering_path.exists() else None
FINAL_SITE = f"resid_post_{N_LAYERS - 1}"
print(f"causal scrub available: {bool(CLASS_KEYS)} | class keys: {CLASS_KEYS}")
print(f"steering prediction available: {STEERING is not None}"
      + (f" | locked betas {sorted(STEERING['predicted'])}" if STEERING else ""))

/content/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 72 | primary seed 103 | identity variable: player
half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651
circuit: ['L0H0', 'L0H1', 'L0MLP', 'L1H0', 'L1H1', 'L1MLP', 'L2H0', 'L2H1', 'L2MLP', 'L3H1', 'L3MLP'] (discovery-selected MB06 candidates + estimation-half exact edges passing BH; exploratory benchmark) | test results already saved: False
causal scrub available: True | class keys: {'L0H0': ['a_ctx', 'team_change', 'player'], 'L0H1': ['player'], 'L0MLP': ['a_ctx', 'team_change', 'player'], 'L1H0': ['a_ctx', 'team_change'], 'L1H1': ['a_ctx', 'team_change'], 'L1MLP': ['a_ctx', 'team_change', 'player'], 'L2H0': ['player'], 'L2H1': ['player'], 'L2MLP': ['a_ctx', 'team_change', 'player'], 'L3H1': ['a_ctx', 'team_change'], 'L3MLP': ['a_ctx', 'team_change', 'player']}
steering prediction available: False


In [4]:
import mi_src.scrub as scrub_tools
import mi_src.steering as steering_tools

def scrub_identity_effect(view, table, components_to_keys, draws, seed0):
    clean, counter, info = scrub_tools.averaged_scrub(MODEL, bundle, view, table, components_to_keys,
        lambda inputs, positions: interventions.remove_feature(inputs, IDENTITY, positions),
        IDENTITY, draws, seed0, BATCH, SCRUB_AT_TARGET)
    return clean - counter

@torch.no_grad()
def steer_delta(view, table, direction, beta):
    """Mean Delta when beta * direction (scaled by the residual norm) is added at the target."""
    unit = torch.as_tensor(np.asarray(direction, dtype=np.float32), device=DEVICE)
    unit = unit / unit.norm()
    out = np.empty(len(table))
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, view, table, BATCH):
        rows = torch.arange(len(positions), device=DEVICE)
        _, cache = hooks.forward(MODEL, inputs, capture=[FINAL_SITE])
        scale = float(STEERING["scale"]) if STEERING else 1.0
        shift = torch.zeros_like(cache[FINAL_SITE])
        shift[rows, positions] = beta * scale * unit
        logits, _ = hooks.forward(MODEL, inputs, replace={FINAL_SITE: hooks.add_at(shift)})
        out[part] = mi_metrics.delta(mi_metrics.at_target(logits, positions), a_ctx, a_id)
    return out


def conflict_tests(view, conflicts, control_table, donor_table):
    """C1-C4 and M1-M3 on one population; returns {test: {estimate, ci_low, ci_high, n, ...}}."""
    out = {}
    matches = conflicts["match_id"].to_numpy()
    delta = circuits.deltas(MODEL, bundle, view, conflicts, BATCH)
    summary = mi_metrics.summarise(delta, matches, N_BOOT)
    out["C1"] = {"estimate": summary["mean"], "ci_low": summary["ci_low"], "ci_high": summary["ci_high"], "n": summary["n"]}

    pplayer = conflicts[conflicts.population == "P-player"].reset_index(drop=True)
    generator = torch.Generator().manual_seed(0)
    c2, c3, c3_matches, c2_matches = [], [], [], []
    with torch.no_grad():
        for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, view, pplayer, BATCH):
            chunk = pplayer.iloc[part]
            base_logits = mi_metrics.at_target(hooks.forward(MODEL, inputs)[0], positions)
            base = mi_metrics.delta(base_logits, a_ctx, a_id)
            own = chunk[IDENTITY].to_numpy()
            if IDENTITY == "player":
                partner = np.array([partner_of.get(p, -1) for p in own])
                valid = partner > 0
                swapped = interventions.set_feature(inputs, "player", positions, torch.as_tensor(np.where(valid, partner, own), device=DEVICE))
                swap_logits = mi_metrics.at_target(hooks.forward(MODEL, swapped)[0], positions)
                swap_change = np.abs(mi_metrics.delta(swap_logits, a_ctx, a_id) - base)
                base_log = mi_metrics.log_softmax(base_logits.cpu().numpy()); swap_log = mi_metrics.log_softmax(swap_logits.cpu().numpy())
                model_change = (swap_log - swap_log.mean(1, keepdims=True)) - (base_log - base_log.mean(1, keepdims=True))
                reference_change = habit_log_probs[np.where(valid, partner, own)] - habit_log_probs[own]
                reference_change = reference_change - reference_change.mean(1, keepdims=True)
                r = (model_change * reference_change).sum(1) / (np.sqrt((model_change ** 2).sum(1) * (reference_change ** 2).sum(1)) + 1e-12)
                c3.append(r[valid]); c3_matches.append(chunk["match_id"].to_numpy()[valid])
            else:
                valid = np.ones(len(chunk), dtype=bool)
                swap_change = np.abs(circuits.delta_under(MODEL, interventions.remove_feature(inputs, IDENTITY, positions),
                                                          positions, a_ctx, a_id) - base)
            vector_change = np.mean([np.abs(circuits.delta_under(MODEL, inputs, positions, a_ctx, a_id,
                                     {f"factor_{IDENTITY}": interventions.norm_matched_vector(None, positions, generator)}) - base)
                                     for _ in range(SETTINGS["n_draws"])], axis=0)
            c2.append((swap_change - vector_change)[valid]); c2_matches.append(chunk["match_id"].to_numpy()[valid])
    for name, values, groups in [("C2", c2, c2_matches), ("C3", c3, c3_matches)]:
        if values:
            summary = mi_metrics.summarise(np.concatenate(values), np.concatenate(groups), N_BOOT)
            out[name] = {"estimate": summary["mean"], "ci_low": summary["ci_low"], "ci_high": summary["ci_high"], "n": summary["n"]}

    if IDENTITY == "player" and len(donor_table):
        units = pplayer.merge(donor_table[["event_row", "donor_window"]], on="event_row")
        units["partner"] = [partner_of.get(p, -1) for p in units["player"]]
        units = units[units["partner"] > 0].reset_index(drop=True)
        frozen_sites = [f"attn_out_{l}" for l in range(N_LAYERS)] + [f"mlp_out_{l}" for l in range(N_LAYERS)]
        differences, groups = [], []
        with torch.no_grad():
            for part in mi_setup.batch_slices(len(units), BATCH):
                chunk = units.iloc[part]
                x_inputs, _ = mi_setup.window_inputs(bundle, view, chunk["window"].to_numpy())
                y_inputs, _ = mi_setup.window_inputs(bundle, view, chunk["donor_window"].to_numpy())
                positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
                a_ctx = torch.as_tensor(chunk["a_ctx"].to_numpy(), device=DEVICE); a_id = torch.as_tensor(chunk["a_id"].to_numpy(), device=DEVICE)
                cells, floor = {}, {}
                for context_name, context_inputs in [("X", x_inputs), ("Y", y_inputs)]:
                    with_a = interventions.set_feature(context_inputs, "player", positions, torch.as_tensor(chunk["player"].to_numpy(), device=DEVICE))
                    with_b = interventions.set_feature(context_inputs, "player", positions, torch.as_tensor(chunk["partner"].to_numpy(), device=DEVICE))
                    logits_a, cache_a = hooks.forward(MODEL, with_a, capture=frozen_sites)
                    cells[("A", context_name)] = mi_metrics.delta(mi_metrics.at_target(logits_a, positions), a_ctx, a_id)
                    cells[("B", context_name)] = circuits.delta_under(MODEL, with_b, positions, a_ctx, a_id)
                    frozen = {site: (lambda value, kept=cache_a[site]: kept) for site in frozen_sites}
                    floor[("B", context_name)] = circuits.delta_under(MODEL, with_b, positions, a_ctx, a_id, frozen)
                interaction = cells[("A", "X")] - cells[("A", "Y")] - cells[("B", "X")] + cells[("B", "Y")]
                interaction_floor = cells[("A", "X")] - cells[("A", "Y")] - floor[("B", "X")] + floor[("B", "Y")]
                differences.append(np.abs(interaction) - np.abs(interaction_floor)); groups.append(chunk["match_id"].to_numpy())
        if differences:
            summary = mi_metrics.summarise(np.concatenate(differences), np.concatenate(groups), N_BOOT)
            out["C4"] = {"estimate": summary["mean"], "ci_low": summary["ci_low"], "ci_high": summary["ci_high"], "n": summary["n"]}

    if CIRCUIT:
        def remove_identity(inputs, positions, chunk):
            return interventions.remove_feature(inputs, IDENTITY, positions)
        p_matches = pplayer["match_id"].to_numpy()
        runs = {}
        for name, ablated in [("full", []), ("removed", CIRCUIT), ("circuit only", [c for c in ALL if c not in CIRCUIT]), ("empty", ALL)]:
            clean, corrupt = circuits.identity_effects(MODEL, bundle, view, pplayer, BATCH, remove_identity,
                                                       circuits.mean_ablation(ablated, MEANS, False))
            runs[name] = clean - corrupt
        summary = mi_metrics.summarise(np.abs(runs["full"]) - np.abs(runs["removed"]), p_matches, N_BOOT)
        out["M1"] = {"estimate": summary["mean"], "ci_low": summary["ci_low"], "ci_high": summary["ci_high"], "n": summary["n"]}
        f = mi_metrics.pooled_recovery(runs["circuit only"] - runs["empty"], runs["full"] - runs["empty"], p_matches, N_BOOT)
        random_f = []
        for components in circuits.enumerate_size_matched_circuits(ALL, CIRCUIT):
            clean, corrupt = circuits.identity_effects(MODEL, bundle, view, pplayer, BATCH, remove_identity,
                                                       circuits.mean_ablation([c for c in ALL if c not in components], MEANS, False))
            random_f.append(mi_metrics.pooled_ratio((clean - corrupt) - runs["empty"], runs["full"] - runs["empty"]))
        out["M2"] = {"estimate": f["recovery"], "ci_low": f["ci_low"], "ci_high": f["ci_high"], "n": len(pplayer),
                     "random_95th_percentile": float(np.nanpercentile(random_f, 95)) if len(random_f) else float("nan"), "n_random": len(random_f)}
        if len(control_table):
            removal = circuits.mean_ablation(CIRCUIT, MEANS, False)
            conflict_change = circuits.deltas(MODEL, bundle, view, pplayer, BATCH, removal) - circuits.deltas(MODEL, bundle, view, pplayer, BATCH)
            control_change = circuits.deltas(MODEL, bundle, view, control_table, BATCH, removal) - circuits.deltas(MODEL, bundle, view, control_table, BATCH)
            paired = pd.DataFrame({"partner_event_row": control_table["partner_event_row"].to_numpy(), "control": control_change}).merge(
                pd.DataFrame({"partner_event_row": pplayer["event_row"].to_numpy(), "conflict": conflict_change, "match_id": p_matches}),
                on="partner_event_row")
            summary = mi_metrics.summarise(np.abs(paired["conflict"]) - np.abs(paired["control"]), paired["match_id"], N_BOOT)
            out["M3"] = {"estimate": summary["mean"], "ci_low": summary["ci_low"], "ci_high": summary["ci_high"], "n": summary["n"]}
    if CIRCUIT and CLASS_KEYS:
        outside = [c for c in ALL if c not in CIRCUIT]
        full = scrub_identity_effect(view, pplayer, {}, 1, 0)
        floor = scrub_identity_effect(view, pplayer, {c: None for c in ALL}, SCRUB_DRAWS, 300)
        consistent = scrub_identity_effect(
            view, pplayer, {**{c: CLASS_KEYS.get(c) for c in CIRCUIT}, **{c: None for c in outside}},
            SCRUB_DRAWS, 310)
        share = mi_metrics.pooled_recovery(consistent - floor, full - floor, pplayer["match_id"].to_numpy(), N_BOOT)
        out["M4"] = {"estimate": share["recovery"], "ci_low": share["ci_low"], "ci_high": share["ci_high"],
                     "n": len(pplayer)}

    if STEERING is not None and IDENTITY == "player":
        direction = STEERING["direction_vector"]
        base = circuits.deltas(MODEL, bundle, view, pplayer, BATCH)
        for number, beta_text in enumerate(sorted(STEERING["predicted"], key=float), start=1):
            beta = float(beta_text)
            change = steer_delta(view, pplayer, direction, beta) - base
            summary = mi_metrics.summarise(change, pplayer["match_id"].to_numpy(), N_BOOT)
            expected = STEERING["predicted"][beta_text]
            out[f"S{number}"] = {"estimate": summary["mean"], "ci_low": summary["ci_low"],
                                 "ci_high": summary["ci_high"], "n": summary["n"], "beta": beta,
                                 "predicted": expected["change_in_delta"], "tolerance": expected["tolerance"]}

    for value in out.values():
        for key in list(value):
            value[key] = float(value[key]) if not isinstance(value[key], (int, np.integer)) else int(value[key])
    return out

In [5]:
validation_path = OUT / "results" / "MB12_validation_estimates.json"
if validation_path.exists():
    validation_estimates = mi_setup.load_json(validation_path)
    print("loaded saved validation estimates")
else:
    val_estimation = val_conflicts[val_conflicts.half == "estimation"].reset_index(drop=True)
    start = time.time()
    validation_estimates = conflict_tests(val_view, val_estimation, controls[controls.half == "estimation"].reset_index(drop=True), donors)
    mi_setup.save_json(validation_path, validation_estimates)
    print(f"computed in {time.time() - start:.0f} s")
print(pd.DataFrame(validation_estimates).T.round(4).to_string())

computed in 4 s
    estimate  ci_low  ci_high       n  random_95th_percentile  n_random
C1    6.2093  6.0768   6.3380  5180.0                     NaN       NaN
C2   -0.0841 -0.1765   0.0319   529.0                     NaN       NaN
C3    0.0758  0.0187   0.1301   529.0                     NaN       NaN
C4    0.7869  0.7171   0.8793   529.0                     NaN       NaN
M1    0.3936  0.3052   0.4714   529.0                     NaN       NaN
M2    1.0114  0.9979   1.0272   529.0                  1.0504       8.0
M3    2.1433  1.4384   3.0411    65.0                     NaN       NaN
M4    0.7469  0.6591   0.8154   529.0                     NaN       NaN


In [6]:
plan_path=OUT/'config/analysis_plan.json'
plan=mi_setup.load_json(plan_path)
prior_plan_hash=fingerprint.file_sha256(plan_path)
adjacency_policy=mi_setup.load_json(OUT/'config/adjacency_policy.json')
assert adjacency_policy['enabled'] is False
fingerprint.record_files(MANIFEST,{'config/adjacency_policy.json':OUT/'config/adjacency_policy.json'})
plan['test_policy']='Retrospective known-test diagnostics on the new wd0.1 benchmark; no fresh held-out confirmation.'
mi_setup.save_json(plan_path,plan)
with (OUT/'manifests/plan_amendments.jsonl').open('a') as handle:
    handle.write(json.dumps({'time':fingerprint.now(),'before_sha256':prior_plan_hash,'after_sha256':fingerprint.file_sha256(plan_path),'change':'Retain MB12 measurements as retrospective known-test diagnostics; no new prospective claim'})+'\n')
fingerprint.record_files(MANIFEST,{'analysis_plan.json':plan_path})
import hashlib
implementation_sources=[cell['source'] for cell in json.loads((MI_DIR/'notebooks/MB12_locked_confirmation.ipynb').read_text())['cells'] if cell['cell_type']=='code']
implementation_code_sha256=hashlib.sha256(json.dumps(implementation_sources,sort_keys=True).encode()).hexdigest()
locked={'status':'RETROSPECTIVE DIAGNOSTIC PLAN; test data and historical outcomes previously accessed',
        'mechanism_statement':'Target-output equivalence-class resampling and component sufficiency; no serial computation established.',
        'analysis_id':plan['analysis_id'],'checkpoint_hashes':plan['checkpoint_hashes'],
        'thresholds':{'C1-C4/M1/M3':'descriptive directional diagnostics; lower match-bootstrap interval > 0',
                      'M2':'component F above distinct size-matched alternative 95th percentile',
                      'M4':'report paired output-resampling recovery, including the >=0.80 hypothesis-preservation bar'},
        'source_hashes':{p.name:fingerprint.file_sha256(p) for p in (MI_DIR/'src').glob('*.py')},
        'implementation_notebook_code_sha256':implementation_code_sha256,
        'written_at':fingerprint.now()}
mi_setup.save_json(OUT/'config/locked_predictions.json',locked)
fingerprint.record_files(MANIFEST,{'config/locked_predictions.json':OUT/'config/locked_predictions.json'})
print(locked['status'])


RETROSPECTIVE DIAGNOSTIC PLAN; test data and historical outcomes previously accessed


In [7]:
if RESULTS_PATH.exists():
    test_results = mi_setup.load_json(RESULTS_PATH)
    print("test results reloaded; the test split was not opened again")
else:
    test_view = mi_setup.unlock_test_view(events, OUT)
    conflict_set = pd.read_parquet(OUT / "artifacts" / "conflict" / "conflict_set.parquet")
    test_conflicts = conflict_set[conflict_set.split == "test"].reset_index(drop=True)
    windows, positions = mi_setup.locate(test_view, test_conflicts["event_row"].to_numpy())
    test_conflicts["window"], test_conflicts["position"] = windows, positions
    test_conflicts["population"] = np.where(test_conflicts["has_dedicated_player"], "P-player", "index-0")
    test_conflicts["half"] = "test"
    if RUN_MODE == "smoke":
        test_conflicts = pd.concat([group.sample(min(len(group), SETTINGS["max_events"]), random_state=0)
                                    for _, group in test_conflicts.groupby("population")]).sort_values("event_row").reset_index(drop=True)

    test_probs = pd.read_parquet(OUT / "artifacts" / "conflict" / "predictions" / "test_probs.parquet")
    context_top = pd.Series(np.stack(test_probs["P_ctx"].to_numpy()).argmax(1), index=test_probs["event_row"].to_numpy())
    pool = pd.DataFrame({"event_row": test_view["event_rows"], "window": test_view["window_index"], "position": test_view["position"]})
    pool = pool[~pool["event_row"].isin(set(conflict_set["event_row"])) & pool["event_row"].isin(context_top.index)]
    pool["match_id"] = bundle["match_ids"][pool["event_row"]]
    pool["player"] = features["player"][pool["event_row"]]
    pool["a_ctx"] = context_top.loc[pool["event_row"]].to_numpy()
    pool["coupled_with_prev"] = events["coupled_with_prev"].to_numpy()[pool["event_row"]]
    pool["team_change"] = events["team_change"].to_numpy()[pool["event_row"]]
    pool["third"] = np.minimum((events["x_norm"].to_numpy()[pool["event_row"]] * 3).astype(int), 2)
    pool = pool[pool["player"] > 0]
    tier_keys = {1: ["player", "a_ctx", "coupled_with_prev", "team_change", "third"], 2: ["player", "a_ctx", "coupled_with_prev"]}
    pool_groups = {tier: {key: group for key, group in pool.groupby(keys)} for tier, keys in tier_keys.items()}
    used, control_rows = set(), []
    for _, row in test_conflicts[test_conflicts.population == "P-player"].iterrows():
        values = {"player": row["player"], "a_ctx": row["a_ctx"], "coupled_with_prev": row["coupled_with_prev"],
                  "team_change": row["team_change"], "third": min(int(events["x_norm"].iat[int(row["event_row"])] * 3), 2)}
        for tier, keys in tier_keys.items():
            options = pool_groups[tier].get(tuple(values[k] for k in keys))
            if options is None:
                continue
            options = options[(options["match_id"] == row["match_id"]) & ~options["event_row"].isin(used)]
            if len(options):
                best = options.iloc[np.argmin(np.abs(options["position"].to_numpy() - row["position"]))]
                used.add(int(best["event_row"]))
                control_rows.append({"event_row": int(best["event_row"]), "window": int(best["window"]), "position": int(best["position"]),
                                     "match_id": int(bundle["match_ids"][int(best["event_row"])]), "a_ctx": int(row["a_ctx"]),
                                     "a_id": int(row["a_id"]), "partner_event_row": int(row["event_row"]), "half": "test", "match_tier": tier})
                break
    test_controls = pd.DataFrame(control_rows)
    if len(test_controls):
        lookup = test_conflicts.set_index("event_row")["match_id"]
        assert (test_controls["match_id"].to_numpy() == lookup.loc[test_controls["partner_event_row"]].to_numpy()).all()
    test_controls.to_parquet(OUT / "artifacts/populations/test_matched_controls.parquet", index=False)

    rng = np.random.default_rng(0)
    p_test = test_conflicts[test_conflicts.population == "P-player"]
    donor_rows = []
    for _, row in p_test.iterrows():
        options = p_test[(p_test["role"] == row["role"]) & (p_test["position"] == row["position"]) & (p_test["match_id"] != row["match_id"])]
        if len(options) == 0:
            options = p_test[(p_test["position"] == row["position"]) & (p_test["match_id"] != row["match_id"])]
        if len(options):
            donor = options.iloc[int(rng.integers(0, len(options)))]
            donor_rows.append({"event_row": int(row["event_row"]), "donor_window": int(donor["window"])})
    test_donors = pd.DataFrame(donor_rows, columns=["event_row", "donor_window"])
    print(f"test conflict rows: {len(test_conflicts):,} ({(test_conflicts.population == 'P-player').sum():,} P-player) | "
          f"matched controls: {len(test_controls):,} | donors: {len(test_donors):,} | matches: {test_conflicts['match_id'].nunique()}")

    start = time.time()
    test_results = conflict_tests(test_view, test_conflicts, test_controls, test_donors)
    test_results["_populations"] = {"n_conflicts": int(len(test_conflicts)), "n_p_player": int((test_conflicts.population == "P-player").sum()),
                                    "n_controls": int(len(test_controls)), "n_matches": int(test_conflicts["match_id"].nunique()),
                                    "p_all_rowset_sha256": fingerprint.rowset_sha256(test_conflicts["event_id"])}
    mi_setup.save_json(RESULTS_PATH, test_results)
    fingerprint.record_files(MANIFEST, {"results/MB12_test_results.json": RESULTS_PATH})
    fingerprint.register(REGISTRY, {"notebook": "MB12", "event": "test evaluated", "seconds": round(time.time() - start)})
    print(f"test evaluated in {time.time() - start:.0f} s")

test conflict rows: 11,101 (1,967 P-player) | matched controls: 283 | donors: 1,964 | matches: 35


test evaluated in 9 s


In [8]:
diagnostic_rows=[]
for name,value in test_results.items():
    if name.startswith('_'): continue
    row={'diagnostic':name, **value, 'evidence_status':'retrospective known-test diagnostic; not confirmation'}
    row['directional rule satisfied']=value.get('estimate',float('nan')) > value.get('random_95th_percentile',0) if name=='M2' else value.get('ci_low',float('nan')) > (0.8 if name=='M4' else 0)
    diagnostic_rows.append(row)
diagnostic_table=pd.DataFrame(diagnostic_rows)
diagnostic_table.to_csv(OUT/'results/MB12_retrospective_diagnostics.csv',index=False)
print(diagnostic_table.to_string(index=False))
fingerprint.record_files(MANIFEST,{'results/MB12_retrospective_diagnostics.csv':OUT/'results/MB12_retrospective_diagnostics.csv'})


diagnostic  estimate    ci_low   ci_high     n                                       evidence_status  directional rule satisfied  random_95th_percentile  n_random
        C1  6.097251  5.952394  6.240140 11101 retrospective known-test diagnostic; not confirmation                        True                     NaN       NaN
        C2 -0.106817 -0.160368 -0.042047  1948 retrospective known-test diagnostic; not confirmation                       False                     NaN       NaN
        C3  0.123443  0.099010  0.148746  1948 retrospective known-test diagnostic; not confirmation                        True                     NaN       NaN
        C4  0.702496  0.638721  0.798346  1945 retrospective known-test diagnostic; not confirmation                        True                     NaN       NaN
        M1  0.368631  0.335036  0.406359  1967 retrospective known-test diagnostic; not confirmation                        True                     NaN       NaN
        M2  0.994030  

{'artifacts/conflict/conflict_set.parquet': {'path': 'artifacts/conflict/conflict_set.parquet',
  'bytes': 1735548,
  'sha256': '19d6197298f38803450ba3befbc0e1b744b6ab900eecb0f6374783c4cf2263b3',
  'recorded_at': '2026-10-05T17:22:56.743259+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt',
  'bytes': 330757,
  'sha256': '590fbc131586f5297fcd9ddfcdf7f2e1c503cf13e9f584142a5c31a10a79339c',
  'recorded_at': '2026-10-05T17:22:56.743711+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt',
  'bytes': 2280753,
  'sha256': 'eab81928ab7743ddbe2cbdb1dc60d5b584408386e896705115145e7c8a6d62a2',
  'recorded_at': '2026-10-05T17:22:56.746435+00:00'},
 'artifacts/conflict/predictions/test_probs.parquet': {'path': 'artifacts/conflict/predictions/test_probs.parquet',
  'bytes': 7982855,
  'sha256': 'd21c146acd76ee17cb82670b

In [9]:
import os, json, sys, importlib.util
from pathlib import Path
import pandas as pd
ROOT=Path(os.environ.get('SOCCER_MI_ROOT',str(ROOT))); MI_DIR=Path(os.environ.get('MI_ANALYSIS_DIR',ROOT/'mi_analysis'))
sys.path.insert(0,str(ROOT/'model_training_evaluation'))
spec=importlib.util.spec_from_file_location('mi_src',MI_DIR/'src/__init__.py',submodule_search_locations=[str(MI_DIR/'src')])
mi_src=importlib.util.module_from_spec(spec);sys.modules['mi_src']=mi_src;spec.loader.exec_module(mi_src)
from mi_src import fingerprint
plan=json.loads((MI_DIR/'config/analysis_plan.json').read_text())
verified=fingerprint.verify_files(MI_DIR/'manifests/sha_manifest.json')
changed=fingerprint.compare_snapshot(MI_DIR/'manifests/track_a_snapshot.json')
assert not changed,changed
for seed in plan['seeds']:
    run=ROOT/'model_training_evaluation/benchmarks/track_a_4layer/runs'/f'transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1_s{seed}'
    assert fingerprint.file_sha256(run/'best.pt')==plan['checkpoint_hashes'][str(seed)]
configs={str(p.relative_to(MI_DIR)):fingerprint.file_sha256(p) for p in (MI_DIR/'config').glob('*.json')}
sources={str(p.relative_to(MI_DIR)):fingerprint.file_sha256(p) for p in (MI_DIR/'src').glob('*.py')}
outputs={}
for name in ['MB03_summary_corrected.json','MB04_summary.json','MB10_summary_corrected.json','MB11_summary_corrected.json','MB11b_summary_corrected.json','MB11c_summary_corrected.json']:
    p=MI_DIR/'results'/name
    assert p.exists(),p
    outputs[name]={'sha256':fingerprint.file_sha256(p),'status':'new benchmark exploratory validation'}
legacy_log=ROOT/'mi_analysis_backup_2026-10-04/logs/test_access_log.jsonl'
legacy_digest=fingerprint.file_sha256(legacy_log) if legacy_log.exists() else None
report={'analysis_id':plan['analysis_id'],'status':'RETROSPECTIVE BENCHMARK DIAGNOSTICS AND PROVENANCE AUDIT COMPLETE; FRESH HELD-OUT CONFIRMATION UNAVAILABLE',
        'reason':'Historical test outcomes were accessed before repairs; rerunning on another checkpoint does not restore unseen data.',
        'new_test_accesses':len((MI_DIR/'logs/test_access_log.jsonl').read_text().splitlines()) if (MI_DIR/'logs/test_access_log.jsonl').exists() else 0,'historical_test_log_sha256':legacy_digest,'verified_manifest_entries':len(verified),
        'track_a_changed':changed,'checkpoint_hashes':plan['checkpoint_hashes'],'config_hashes':configs,'source_hashes':sources,'validation_outputs':outputs,
        'M4':'output-equivalence resampling only; no serial computation established','steering':'exploratory population transfer and unseen-dose interpolation, not locked test confirmation'}
p=MI_DIR/'results/MB12_validation_audit.json';p.write_text(json.dumps(report,indent=2));fingerprint.record_files(MI_DIR/'manifests/sha_manifest.json',{'results/MB12_validation_audit.json':p})
print(json.dumps(report,indent=2))


{
  "analysis_id": "benchmark_4L_2H_K16_wd0.1_2026-10-05",
  "status": "RETROSPECTIVE BENCHMARK DIAGNOSTICS AND PROVENANCE AUDIT COMPLETE; FRESH HELD-OUT CONFIRMATION UNAVAILABLE",
  "reason": "Historical test outcomes were accessed before repairs; rerunning on another checkpoint does not restore unseen data.",
  "new_test_accesses": 1,
  "historical_test_log_sha256": null,
  "verified_manifest_entries": 76,
  "track_a_changed": [],
  "checkpoint_hashes": {
    "100": "d9712d0a305481eeac0d110f561677fa50838809d1cf84f0203cabc20a96358f",
    "101": "1b04a5520e2180258456ad3574f71d3eb37d3049da29960e88ea59623ac290a3",
    "102": "56134ec75d1d09b3451bab2a0857ee5cc922c7d5c2ea8d5534db2cb3736b19a9",
    "103": "364652fec05af19d4b26fd10af56f98e4f3ae7ef68f60340d67b7bed2bd19d48",
    "104": "e0b89ecde7df3a6b6ad0bbc48e7f48d85638949173f0c5b318d9644bb244a446"
  },
  "config_hashes": {
    "config/analysis_plan.json": "0be22a397ce1ac6a0d3644cfa8d4dbf5f4532db95172eaf90a0ac41fde531f3a",
    "config/locke